In [1]:
import numpy as np
import pandas as pd
from pyhive import presto
from datetime import timedelta,datetime
import json
import requests
import math
import os


pd.set_option('display.max_columns', None)
import warnings
warnings.filterwarnings("ignore")
conn = presto.connect(
        host="presto-gateway.serving.data.plectrum.dev",
        port=80,
        username="manoj.ravirajan@rapido.bike"
    )
presto_port = '80'
username = 'manoj.ravirajan@rapido.bike'
conn1 = presto.connect('bi-presto.serving.data.production.internal', presto_port, username)
conn2 = presto.connect('bi-trino-2.serving.data.production.internal', presto_port, username)
conn3= presto.connect('processing-2.processing.data.production.internal',presto_port,username)
conn4= presto.connect('processing.processing.data.production.internal',presto_port,username)

### new segment

In [48]:
cx_data0 = f'''
    with base as (
    select 
        customer_id, 
        segment  
    from 
        hive.experiments_internal.customer_segment_amj25
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        count(distinct order_id) orders
    from 
        orders.order_logs_fact
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        -- and order_status='dropped'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    )
    
    
    select
        segment,
        sum(case when week_start_date = '20241216' then orders end ) w01,
        sum(case when week_start_date = '20241223' then orders end ) w02,
        sum(case when week_start_date = '20241230' then orders end ) w03,
        sum(case when week_start_date = '20250106' then orders end ) w04,
        sum(case when week_start_date = '20250113' then orders end ) w05,
        sum(case when week_start_date = '20250120' then orders end ) w06,
        sum(case when week_start_date = '20250127' then orders end ) w07,
        sum(case when week_start_date = '20250203' then orders end ) w08,
        sum(case when week_start_date = '20250210' then orders end ) w09,
        sum(case when week_start_date = '20250217' then orders end ) w10,
        sum(case when week_start_date = '20250224' then orders end ) w11,
        sum(case when week_start_date = '20250303' then orders end ) w12,
        sum(case when week_start_date = '20250310' then orders end ) w13,
        
        sum(orders.orders) as overall,
        100.00*sum(orders)/sum(sum(orders))over() as pct,
        count(distinct case when week_start_date = '20241216' then orders.customer_id end ) C_w01,
        count(distinct case when week_start_date = '20241223' then orders.customer_id end ) C_w02,
        count(distinct case when week_start_date = '20241230' then orders.customer_id end ) C_w03,
        count(distinct case when week_start_date = '20250106' then orders.customer_id end ) C_w04,
        count(distinct case when week_start_date = '20250113' then orders.customer_id end ) C_w05,
        count(distinct case when week_start_date = '20250120' then orders.customer_id end ) C_w06,
        count(distinct case when week_start_date = '20250127' then orders.customer_id end ) C_w07,
        count(distinct case when week_start_date = '20250203' then orders.customer_id end ) C_w08,
        count(distinct case when week_start_date = '20250210' then orders.customer_id end ) C_w09,
        count(distinct case when week_start_date = '20250217' then orders.customer_id end ) C_w10,
        count(distinct case when week_start_date = '20250224' then orders.customer_id end ) C_w11,
        count(distinct case when week_start_date = '20250303' then orders.customer_id end ) C_w12,
        count(distinct case when week_start_date = '20250310' then orders.customer_id end ) C_w13,
        
        count(distinct orders.customer_id) as C_overall,
        100.00*count(distinct orders.customer_id)/sum(count(distinct orders.customer_id))over() as c_pct

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1
    order by 1
       
'''
cx_data0 = pd.read_sql(cx_data0, conn4)
cx_data0.head()

,segment,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,1-DAILY,978303,732304,828267,1104298,1015273,1197754,1172648,1252083,1281503,1136586,1117268,1189747,1086237,14092271,22.56,99558,91818,97572,104353,104487,106050,106339,106220,106016,105406,104797,104043,102128,108875,2.41
1,2-WEEKLY,1029757,801451,901391,1110973,1032780,1185439,1157025,1253815,1296343,1136923,1131776,1201827,1133625,14373125,23.01,243502,219280,235847,258073,254294,260574,261071,264211,264497,259163,257990,255597,251062,296368,6.55
2,3-MONTHLY,1912620,1540358,1708287,1950871,1935235,2152872,2150166,2496496,2492256,2093442,2095683,2220009,2203215,26951510,43.15,733668,643069,699252,759550,770624,801263,814496,895653,864142,782717,779494,774980,781254,2171381,47.99
3,4-OTHER,421694,342426,332751,280318,254822,304261,301702,313374,477777,549158,619018,745326,810159,5752786,9.21,184880,160899,159161,140535,132036,145936,148239,154483,218064,236952,259337,283813,309345,1794354,39.66
4,5-INACTIVE,93133,77267,86830,91177,88082,98514,99608,111225,114783,103540,105653,111969,112330,1294111,2.07,34010,31147,34306,32205,31995,33171,34763,36752,37579,35899,36581,37313,38637,153277,3.39


In [7]:
cx_data0

,segment,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,1-DAILY,525448,410183,462315,583872,539047,620750,609391,645308,660706,590417,584798,614802,566184,7413221,11.87,43406,41360,43308,44635,44498,44725,44790,44749,44710,44639,44515,44232,43635,45183,1.00
1,2-WEEKLY,1468247,1111724,1254550,1614265,1487610,1730882,1687116,1824895,1882465,1654254,1635418,1746836,1626346,20724608,33.18,298500,268749,289092,316525,312343,319525,320119,323232,323496,317833,316214,313366,307589,357293,7.90
2,3-MONTHLY,1926985,1552206,1721080,1968005,1956631,2184433,2183332,2532191,2526931,2122280,2124511,2249945,2230547,27279077,43.67,734822,644058,700271,760816,772564,803637,816997,898103,866449,784814,781552,777022,783220,2174148,48.06
3,4-OTHER,421694,342426,332751,280318,254822,304261,301702,313374,477777,549158,619018,745326,810159,5752786,9.21,184880,160899,159161,140535,132036,145936,148239,154483,218064,236952,259337,283813,309345,1794354,39.66
4,5-INACTIVE,93133,77267,86830,91177,88082,98514,99608,111225,114783,103540,105653,111969,112330,1294111,2.07,34010,31147,34306,32205,31995,33171,34763,36752,37579,35899,36581,37313,38637,153277,3.39


In [49]:
cx_data0.to_clipboard(index=False)

### Office bins

In [6]:
cx_data15 = f'''
    with base as (
    select
        customer_id,
        segment,
        -- total_net_orders,
        -- office_orders,
        /*
        case 
        when office_orders = '0' then 'a. 0 office ride'
        when office_orders = '1' then 'b. 1 office ride'
        when office_orders >= '2' and office_orders <= '3' then 'c. 2-3 office'
        when office_orders >= '4' and office_orders <= '6' then 'd. 4-6 office'
        when office_orders >= '7' and office_orders <= '10' then 'e. 7-10 office'
        when office_orders > '10' then 'f. 10+ office'
        else 'a. 0 office ride'
        end office_bin
        
        case 
        when office_orders = '0' and transit_orders = '0' then 'a. no-office-transit'
        when office_orders = '1' then 'b. 1 office ride'
        when office_orders >= '2' and office_orders <= '3' then 'c. 2-3 office'
        when office_orders >= '4' and office_orders <= '6' then 'd. 4-6 office'
        when office_orders >= '7' and office_orders <= '10' then 'e. 7-10 office'
        when office_orders > '10' then 'f. 10+ office'
        when transit_orders = '1' then 'g. 1 transit ride'
        when transit_orders >= '2' and transit_orders <= '3' then 'h. 2-3 transit ride'
        when transit_orders > '3' then 'i. 3+ transit ride'
        else 'a. no-office-transit'
        end office_bin,
        */
        
        case 
        when office_orders = '0' and transit_orders = '0' then 'a. no-office-transit'
        when office_orders >= '1' then 'b. office ride'
        when transit_orders >= '1' then 'c. transit ride'
        else 'a. no-office-transit'
        end office_bin
        
    from 
        hive.experiments_internal.customer_base_office_transit_tag_amj2025
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        count(distinct order_id) orders
    from 
        orders.order_logs_fact
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        -- and order_status='dropped'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    )
    
    
    select
        office_bin,
        sum(case when week_start_date = '20241216' then orders end ) w01,
        sum(case when week_start_date = '20241223' then orders end ) w02,
        sum(case when week_start_date = '20241230' then orders end ) w03,
        sum(case when week_start_date = '20250106' then orders end ) w04,
        sum(case when week_start_date = '20250113' then orders end ) w05,
        sum(case when week_start_date = '20250120' then orders end ) w06,
        sum(case when week_start_date = '20250127' then orders end ) w07,
        sum(case when week_start_date = '20250203' then orders end ) w08,
        sum(case when week_start_date = '20250210' then orders end ) w09,
        sum(case when week_start_date = '20250217' then orders end ) w10,
        sum(case when week_start_date = '20250224' then orders end ) w11,
        sum(case when week_start_date = '20250303' then orders end ) w12,
        sum(case when week_start_date = '20250310' then orders end ) w13,
        
        sum(orders.orders) as overall,
        100.00*sum(orders)/sum(sum(orders))over() as pct,
        count(distinct case when week_start_date = '20241216' then orders.customer_id end ) C_w01,
        count(distinct case when week_start_date = '20241223' then orders.customer_id end ) C_w02,
        count(distinct case when week_start_date = '20241230' then orders.customer_id end ) C_w03,
        count(distinct case when week_start_date = '20250106' then orders.customer_id end ) C_w04,
        count(distinct case when week_start_date = '20250113' then orders.customer_id end ) C_w05,
        count(distinct case when week_start_date = '20250120' then orders.customer_id end ) C_w06,
        count(distinct case when week_start_date = '20250127' then orders.customer_id end ) C_w07,
        count(distinct case when week_start_date = '20250203' then orders.customer_id end ) C_w08,
        count(distinct case when week_start_date = '20250210' then orders.customer_id end ) C_w09,
        count(distinct case when week_start_date = '20250217' then orders.customer_id end ) C_w10,
        count(distinct case when week_start_date = '20250224' then orders.customer_id end ) C_w11,
        count(distinct case when week_start_date = '20250303' then orders.customer_id end ) C_w12,
        count(distinct case when week_start_date = '20250310' then orders.customer_id end ) C_w13,
        
        count(distinct orders.customer_id) as C_overall,
        100.00*count(distinct orders.customer_id)/sum(count(distinct orders.customer_id))over() as c_pct

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1
    order by 1
       
'''
cx_data15 = pd.read_sql(cx_data15, conn3)
cx_data15.head()

,office_bin,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,a. no-office-transit,1095585,889043,962759,1030102,1022487,1114597,1105287,1258681,1317859,1170104,1197008,1289719,1350695,14803926,23.70,452869,399985,424070,430604,435780,450208,456161,500858,516095,482961,493881,503891,531367,2300919,50.86
1,b. office ride,2704099,2063373,2296376,2869861,2664161,3140224,3094706,3411848,3562212,3148195,3160640,3430583,3252421,38798699,62.11,629559,548000,590367,649240,638578,675546,684363,717219,730369,704722,708519,716929,713705,1496901,33.09
2,c. transit ride,635823,541390,598391,637674,639544,684019,681156,756464,782591,701350,711750,748576,742450,8861178,14.19,213190,198228,211701,214872,219078,221240,224384,239242,243834,232454,235799,234926,237354,726435,16.06


In [7]:
cx_data15

,office_bin,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,a. no-office-transit,1095585,889043,962759,1030102,1022487,1114597,1105287,1258681,1317859,1170104,1197008,1289719,1350695,14803926,23.70,452869,399985,424070,430604,435780,450208,456161,500858,516095,482961,493881,503891,531367,2300919,50.86
1,b. office ride,2704099,2063373,2296376,2869861,2664161,3140224,3094706,3411848,3562212,3148195,3160640,3430583,3252421,38798699,62.11,629559,548000,590367,649240,638578,675546,684363,717219,730369,704722,708519,716929,713705,1496901,33.09
2,c. transit ride,635823,541390,598391,637674,639544,684019,681156,756464,782591,701350,711750,748576,742450,8861178,14.19,213190,198228,211701,214872,219078,221240,224384,239242,243834,232454,235799,234926,237354,726435,16.06


In [10]:
cx_data15.to_clipboard(index=False)

In [8]:
cx_data16 = f'''
    with base as (
    select
        customer_id,
        segment,
        -- total_net_orders,
        -- office_orders,
        /*
        case 
        when office_orders = '0' then 'a. 0 office ride'
        when office_orders = '1' then 'b. 1 office ride'
        when office_orders >= '2' and office_orders <= '3' then 'c. 2-3 office'
        when office_orders >= '4' and office_orders <= '6' then 'd. 4-6 office'
        when office_orders >= '7' and office_orders <= '10' then 'e. 7-10 office'
        when office_orders > '10' then 'f. 10+ office'
        else 'a. 0 office ride'
        end office_bin

        case 
        when office_orders = '0' and transit_orders = '0' then 'a. no-office-transit'
        when office_orders = '1' then 'b. 1 office ride'
        when office_orders >= '2' and office_orders <= '3' then 'c. 2-3 office'
        when office_orders >= '4' and office_orders <= '6' then 'd. 4-6 office'
        when office_orders >= '7' and office_orders <= '10' then 'e. 7-10 office'
        when office_orders > '10' then 'f. 10+ office'
        when transit_orders = '1' then 'g. 1 transit ride'
        when transit_orders >= '2' and transit_orders <= '3' then 'h. 2-3 transit ride'
        when transit_orders > '3' then 'i. 3+ transit ride'
        else 'a. no-office-transit'
        end office_bin
        */
        
        case 
        when office_orders = '0' and transit_orders = '0' then 'a. no-office-transit'
        when office_orders >= '1' then 'b. office ride'
        when transit_orders >= '1' then 'c. transit ride'
        else 'a. no-office-transit'
        end office_bin
        
    from 
        hive.experiments_internal.customer_base_office_transit_tag_amj2025
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        count(distinct order_id) orders
    from 
        orders.order_logs_fact
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        -- and order_status='dropped'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    )
    
    
    select
        segment,
        office_bin,
        sum(case when week_start_date = '20241216' then orders end ) w01,
        sum(case when week_start_date = '20241223' then orders end ) w02,
        sum(case when week_start_date = '20241230' then orders end ) w03,
        sum(case when week_start_date = '20250106' then orders end ) w04,
        sum(case when week_start_date = '20250113' then orders end ) w05,
        sum(case when week_start_date = '20250120' then orders end ) w06,
        sum(case when week_start_date = '20250127' then orders end ) w07,
        sum(case when week_start_date = '20250203' then orders end ) w08,
        sum(case when week_start_date = '20250210' then orders end ) w09,
        sum(case when week_start_date = '20250217' then orders end ) w10,
        sum(case when week_start_date = '20250224' then orders end ) w11,
        sum(case when week_start_date = '20250303' then orders end ) w12,
        sum(case when week_start_date = '20250310' then orders end ) w13,
        
        sum(orders.orders) as overall,
        100.00*sum(orders)/sum(sum(orders))over() as pct,
        count(distinct case when week_start_date = '20241216' then orders.customer_id end ) C_w01,
        count(distinct case when week_start_date = '20241223' then orders.customer_id end ) C_w02,
        count(distinct case when week_start_date = '20241230' then orders.customer_id end ) C_w03,
        count(distinct case when week_start_date = '20250106' then orders.customer_id end ) C_w04,
        count(distinct case when week_start_date = '20250113' then orders.customer_id end ) C_w05,
        count(distinct case when week_start_date = '20250120' then orders.customer_id end ) C_w06,
        count(distinct case when week_start_date = '20250127' then orders.customer_id end ) C_w07,
        count(distinct case when week_start_date = '20250203' then orders.customer_id end ) C_w08,
        count(distinct case when week_start_date = '20250210' then orders.customer_id end ) C_w09,
        count(distinct case when week_start_date = '20250217' then orders.customer_id end ) C_w10,
        count(distinct case when week_start_date = '20250224' then orders.customer_id end ) C_w11,
        count(distinct case when week_start_date = '20250303' then orders.customer_id end ) C_w12,
        count(distinct case when week_start_date = '20250310' then orders.customer_id end ) C_w13,
        
        count(distinct orders.customer_id) as C_overall,
        100.00*count(distinct orders.customer_id)/sum(count(distinct orders.customer_id))over() as c_pct

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1,2
    order by 1,2
       
'''
cx_data16 = pd.read_sql(cx_data16, conn3)
cx_data16.head()

,segment,office_bin,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,1-DAILY,a. no-office-transit,85815,61744,72661,96535,90346,106859,104604,110044,111764,100116,96383,102946,95000,1234817,1.98,9908,8996,9804,10520,10600,10804,10812,10776,10741,10646,10568,10452,10272,11040,0.24
1,1-DAILY,b. office ride,806344,603605,677696,910344,833468,984865,964268,1031829,1057744,935267,923142,982890,894200,11605662,18.58,80049,73837,78155,83743,83761,84951,85237,85135,85011,84577,84099,83600,82009,87315,1.93
2,1-DAILY,c. transit ride,86144,66955,77910,97419,91459,106030,103776,110210,111995,101203,97743,103911,97037,1251792,2.00,9601,8985,9613,10090,10126,10295,10290,10309,10264,10183,10130,9991,9847,10520,0.23
3,2-WEEKLY,a. no-office-transit,140647,110569,123338,147285,141163,156558,152257,166182,169367,150812,147996,154745,149910,1910829,3.06,37243,33937,36441,39441,39252,39858,39843,40625,40477,39661,39324,38663,38099,45811,1.01
4,2-WEEKLY,b. office ride,758919,581516,653218,824932,756374,882601,859922,932532,967540,845658,842091,900429,843282,10649014,17.05,172479,153542,165263,182907,179441,184577,184955,186853,187274,183619,182877,181672,178319,208987,4.62


In [9]:
cx_data16

,segment,office_bin,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,1-DAILY,a. no-office-transit,85815,61744,72661,96535,90346,106859,104604,110044,111764,100116,96383,102946,95000,1234817,1.98,9908,8996,9804,10520,10600,10804,10812,10776,10741,10646,10568,10452,10272,11040,0.24
1,1-DAILY,b. office ride,806344,603605,677696,910344,833468,984865,964268,1031829,1057744,935267,923142,982890,894200,11605662,18.58,80049,73837,78155,83743,83761,84951,85237,85135,85011,84577,84099,83600,82009,87315,1.93
2,1-DAILY,c. transit ride,86144,66955,77910,97419,91459,106030,103776,110210,111995,101203,97743,103911,97037,1251792,2.00,9601,8985,9613,10090,10126,10295,10290,10309,10264,10183,10130,9991,9847,10520,0.23
3,2-WEEKLY,a. no-office-transit,140647,110569,123338,147285,141163,156558,152257,166182,169367,150812,147996,154745,149910,1910829,3.06,37243,33937,36441,39441,39252,39858,39843,40625,40477,39661,39324,38663,38099,45811,1.01
4,2-WEEKLY,b. office ride,758919,581516,653218,824932,756374,882601,859922,932532,967540,845658,842091,900429,843282,10649014,17.05,172479,153542,165263,182907,179441,184577,184955,186853,187274,183619,182877,181672,178319,208987,4.62
5,2-WEEKLY,c. transit ride,130191,109366,124835,138756,135243,146280,144846,155101,159436,140453,141689,146653,140433,1813282,2.90,33780,31801,34143,35725,35601,36139,36273,36733,36746,35883,35789,35262,34644,41570,0.92
6,3-MONTHLY,a. no-office-transit,595706,494568,543534,595880,610153,645838,643367,760411,741312,610153,609554,631154,648507,8130137,13.02,267785,237753,257448,274995,283855,289020,292841,329470,311917,274739,273119,269111,275464,917705,20.28
7,3-MONTHLY,b. office ride,977681,751195,837618,1012808,968204,1138599,1137197,1309833,1330112,1125928,1124868,1216876,1183811,14114730,22.60,328711,278368,304019,342345,338931,364305,371420,402158,394087,363773,361879,364225,363511,849541,18.78
8,3-MONTHLY,c. transit ride,339233,294595,327135,342183,356878,368435,369602,426252,420832,357361,361261,371979,370897,4706643,7.53,137172,126948,137785,142210,147838,147938,150235,164025,158138,144205,144496,141644,142279,404135,8.93
9,4-OTHER,a. no-office-transit,243449,196096,193917,163871,154787,177080,175782,189271,261170,278135,311006,366532,420723,3131819,5.01,123013,105642,105083,92925,89372,97351,98530,105048,137541,143330,155921,170151,190845,1236807,27.34


In [11]:
cx_data16.to_clipboard(index=False)

In [20]:
cx_data01 = f'''
    with base as (
    select 
        customer_id, 
        segment  
    from 
        hive.experiments_internal.customer_segment_amj25
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        count(distinct order_id) orders
    from 
        orders.order_logs_fact
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        -- and order_status = 'dropped'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    ),
    
    appo as (
    
    select
        userid,
            case 
                when regexp_like(apps_installed, 'jira|miro|zoho|teams||intune|figma|slack|confluence|webex|github') = true 
                then 'office-apps' 
            else 'no-office-apps' end office_tag,  
        
        CASE 
            WHEN apps_installed LIKE '%uber%' OR apps_installed LIKE '%ola%' OR apps_installed LIKE '%blusmart%' OR apps_installed LIKE '%nammayatri%' OR apps_installed LIKE '%in drive%' 
            OR apps_installed LIKE '%quick ride%' OR apps_installed LIKE '%jugnoo%' 
        THEN 'RHA' 
            else 'NRHA' end as RHA_Check,

        CASE 
            when apps_installed LIKE '%zomato%' OR apps_installed LIKE '%swiggy%' 
        THEN 'Zwigato' 
            else 'Non-Zwigato' end as Zwigato_check

            
    from 
        datasets.customer_apps_installed_snapshot
    )
    
    
    select
        segment,
        coalesce(RHA_Check, 'UNKNOWN') RHA_Check,
        sum(case when week_start_date = '20241216' then orders end ) w01,
        sum(case when week_start_date = '20241223' then orders end ) w02,
        sum(case when week_start_date = '20241230' then orders end ) w03,
        sum(case when week_start_date = '20250106' then orders end ) w04,
        sum(case when week_start_date = '20250113' then orders end ) w05,
        sum(case when week_start_date = '20250120' then orders end ) w06,
        sum(case when week_start_date = '20250127' then orders end ) w07,
        sum(case when week_start_date = '20250203' then orders end ) w08,
        sum(case when week_start_date = '20250210' then orders end ) w09,
        sum(case when week_start_date = '20250217' then orders end ) w10,
        sum(case when week_start_date = '20250224' then orders end ) w11,
        sum(case when week_start_date = '20250303' then orders end ) w12,
        sum(case when week_start_date = '20250310' then orders end ) w13,
        
        sum(orders.orders) as overall,
        100.00*sum(orders)/sum(sum(orders))over() as pct,
        
        count(distinct case when week_start_date = '20241216' then orders.customer_id end ) C_w01,
        count(distinct case when week_start_date = '20241223' then orders.customer_id end ) C_w02,
        count(distinct case when week_start_date = '20241230' then orders.customer_id end ) C_w03,
        count(distinct case when week_start_date = '20250106' then orders.customer_id end ) C_w04,
        count(distinct case when week_start_date = '20250113' then orders.customer_id end ) C_w05,
        count(distinct case when week_start_date = '20250120' then orders.customer_id end ) C_w06,
        count(distinct case when week_start_date = '20250127' then orders.customer_id end ) C_w07,
        count(distinct case when week_start_date = '20250203' then orders.customer_id end ) C_w08,
        count(distinct case when week_start_date = '20250210' then orders.customer_id end ) C_w09,
        count(distinct case when week_start_date = '20250217' then orders.customer_id end ) C_w10,
        count(distinct case when week_start_date = '20250224' then orders.customer_id end ) C_w11,
        count(distinct case when week_start_date = '20250303' then orders.customer_id end ) C_w12,
        count(distinct case when week_start_date = '20250310' then orders.customer_id end ) C_w13,
        
        count(distinct orders.customer_id) as C_overall,
        100.00*count(distinct orders.customer_id)/sum(count(distinct orders.customer_id))over() as c_pct
      
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    left join
        appo
        on base.customer_id = appo.userid
    group by 1,2
    order by 1,2
'''
cx_data11 = pd.read_sql(cx_data01, conn1)
cx_data11.head()

,segment,RHA_Check,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,1-DAILY,NRHA,62796,54607,61135,70769,68221,75441,73977,78978,81685,72816,71882,74706,69023,916036,1.47,5369,5347,5500,5569,5568,5599,5598,5590,5594,5583,5539,5485,5395,5650,0.12
1,1-DAILY,RHA,319145,249438,282990,353098,324677,376262,367977,390780,398452,355609,352433,370688,340591,4482140,7.18,26442,25319,26409,27111,26991,27159,27174,27163,27132,27092,27013,26848,26477,27408,0.61
2,1-DAILY,UNKNOWN,143507,106138,118190,160005,146149,169047,167437,175550,180569,161992,160483,169408,156570,2015045,3.23,11595,10694,11399,11955,11939,11967,12018,11996,11984,11964,11963,11899,11763,12125,0.27
3,2-WEEKLY,NRHA,197621,171431,192919,218720,210282,230720,227441,245750,251781,221870,219622,229558,216165,2833880,4.54,42801,41727,44099,45908,45626,46115,46326,46866,46825,45959,45597,44831,43959,52343,1.16
4,2-WEEKLY,RHA,885261,668955,756295,967509,890082,1038914,1011231,1094943,1127670,991177,982143,1047562,971945,12433687,19.91,180536,162419,175057,190952,188281,192957,193106,195207,195181,191766,191060,189205,185556,215551,4.76


In [21]:
cx_data11

,segment,RHA_Check,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,1-DAILY,NRHA,62796,54607,61135,70769,68221,75441,73977,78978,81685,72816,71882,74706,69023,916036,1.47,5369,5347,5500,5569,5568,5599,5598,5590,5594,5583,5539,5485,5395,5650,0.12
1,1-DAILY,RHA,319145,249438,282990,353098,324677,376262,367977,390780,398452,355609,352433,370688,340591,4482140,7.18,26442,25319,26409,27111,26991,27159,27174,27163,27132,27092,27013,26848,26477,27408,0.61
2,1-DAILY,UNKNOWN,143507,106138,118190,160005,146149,169047,167437,175550,180569,161992,160483,169408,156570,2015045,3.23,11595,10694,11399,11955,11939,11967,12018,11996,11984,11964,11963,11899,11763,12125,0.27
3,2-WEEKLY,NRHA,197621,171431,192919,218720,210282,230720,227441,245750,251781,221870,219622,229558,216165,2833880,4.54,42801,41727,44099,45908,45626,46115,46326,46866,46825,45959,45597,44831,43959,52343,1.16
4,2-WEEKLY,RHA,885261,668955,756295,967509,890082,1038914,1011231,1094943,1127670,991177,982143,1047562,971945,12433687,19.91,180536,162419,175057,190952,188281,192957,193106,195207,195181,191766,191060,189205,185556,215551,4.76
5,2-WEEKLY,UNKNOWN,385365,271338,305336,428036,387246,461248,448444,484202,503014,441207,433653,469716,438236,5457041,8.74,75163,64603,69936,79665,78436,80453,80687,81159,81490,80108,79557,79330,78074,89399,1.98
6,3-MONTHLY,NRHA,430050,382115,428973,464098,477900,488239,488016,571638,563494,466460,467445,482938,493598,6204964,9.93,164479,157254,171150,181385,188341,185015,187235,208749,198775,178188,177768,174000,178526,543606,12.02
7,3-MONTHLY,RHA,1113238,892456,982005,1117884,1110028,1255306,1252006,1447100,1445438,1217190,1224600,1298156,1275440,15630847,25.02,428238,372779,403760,436540,442618,464755,472519,516794,499191,453389,453683,450933,452143,1238224,27.37
8,3-MONTHLY,UNKNOWN,383697,277635,310102,386023,368703,440888,443310,513453,517999,438630,432466,468851,461509,5443266,8.71,142105,114025,125361,142891,141605,153867,157243,172560,168483,153237,150101,152089,152551,392318,8.67
9,4-OTHER,NRHA,132965,112717,111916,95647,89191,100260,99165,103159,150834,170469,190522,225139,249110,1831094,2.93,58517,53880,53522,48201,46067,48111,49053,50881,69961,76125,83470,91002,100233,608602,13.45


In [24]:
cx_data11.to_clipboard(index=False)

In [22]:
cx_data02 = f'''
    with base as (
    select 
        customer_id, 
        segment  
    from 
        hive.experiments_internal.customer_segment_amj25
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        count(distinct order_id) orders
    from 
        orders.order_logs_fact
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        -- and order_status = 'dropped'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    ),
    
    appo as (
    
    select
        userid,
            /* 
            case 
                when regexp_like(apps_installed, 'jira|miro|zoho|teams||intune|figma|slack|confluence|webex|github') = true 
                then 'office-apps' 
            else 'no-office-apps' end office_tag,  
        */
        CASE 
            WHEN apps_installed LIKE '%uber%' OR apps_installed LIKE '%ola%' OR apps_installed LIKE '%blusmart%' OR apps_installed LIKE '%nammayatri%' OR apps_installed LIKE '%in drive%' 
            OR apps_installed LIKE '%quick ride%' OR apps_installed LIKE '%jugnoo%' 
        THEN 'RHA' 
            else 'NRHA' end as RHA_Check,
        
        
        CASE 
            when apps_installed LIKE '%zomato%' OR apps_installed LIKE '%swiggy%' 
        THEN 'Zwigato' 
            else 'Non-Zwigato' end as Zwigato_check

            
    from 
        datasets.customer_apps_installed_snapshot
    )
    
    
    select
        segment,
        coalesce(Zwigato_check, 'UNKNOWN') Zwigato_check,
        sum(case when week_start_date = '20241216' then orders end ) w01,
        sum(case when week_start_date = '20241223' then orders end ) w02,
        sum(case when week_start_date = '20241230' then orders end ) w03,
        sum(case when week_start_date = '20250106' then orders end ) w04,
        sum(case when week_start_date = '20250113' then orders end ) w05,
        sum(case when week_start_date = '20250120' then orders end ) w06,
        sum(case when week_start_date = '20250127' then orders end ) w07,
        sum(case when week_start_date = '20250203' then orders end ) w08,
        sum(case when week_start_date = '20250210' then orders end ) w09,
        sum(case when week_start_date = '20250217' then orders end ) w10,
        sum(case when week_start_date = '20250224' then orders end ) w11,
        sum(case when week_start_date = '20250303' then orders end ) w12,
        sum(case when week_start_date = '20250310' then orders end ) w13,

        sum(orders.orders) as overall,
        100.00*sum(orders)/sum(sum(orders))over() as pct,
        
        count(distinct case when week_start_date = '20241216' then orders.customer_id end ) C_w01,
        count(distinct case when week_start_date = '20241223' then orders.customer_id end ) C_w02,
        count(distinct case when week_start_date = '20241230' then orders.customer_id end ) C_w03,
        count(distinct case when week_start_date = '20250106' then orders.customer_id end ) C_w04,
        count(distinct case when week_start_date = '20250113' then orders.customer_id end ) C_w05,
        count(distinct case when week_start_date = '20250120' then orders.customer_id end ) C_w06,
        count(distinct case when week_start_date = '20250127' then orders.customer_id end ) C_w07,
        count(distinct case when week_start_date = '20250203' then orders.customer_id end ) C_w08,
        count(distinct case when week_start_date = '20250210' then orders.customer_id end ) C_w09,
        count(distinct case when week_start_date = '20250217' then orders.customer_id end ) C_w10,
        count(distinct case when week_start_date = '20250224' then orders.customer_id end ) C_w11,
        count(distinct case when week_start_date = '20250303' then orders.customer_id end ) C_w12,
        count(distinct case when week_start_date = '20250310' then orders.customer_id end ) C_w13,
        
        count(distinct orders.customer_id) as C_overall,
        100.00*count(distinct orders.customer_id)/sum(count(distinct orders.customer_id))over() as c_pct

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    left join
        appo
        on base.customer_id = appo.userid
    group by 1,2
    order by 1,2
'''
cx_data12 = pd.read_sql(cx_data02, conn1)
cx_data12.head()

,segment,Zwigato_check,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,1-DAILY,Non-Zwigato,81265,70487,78228,90873,86702,97732,96145,101465,103863,92928,91658,94748,87363,1173457,1.88,6827,6787,6977,7092,7087,7142,7127,7122,7108,7108,7051,6960,6856,7185,0.16
1,1-DAILY,UNKNOWN,143507,106138,118190,160005,146149,169047,167437,175550,180569,161992,160483,169408,156570,2015045,3.23,11595,10694,11399,11955,11939,11967,12018,11996,11984,11964,11963,11899,11763,12125,0.27
2,1-DAILY,Zwigato,300676,233558,265897,332994,306196,353971,345809,368293,376274,335497,332657,350646,322251,4224719,6.76,24984,23879,24932,25588,25472,25616,25645,25631,25618,25567,25501,25373,25016,25873,0.57
3,2-WEEKLY,Non-Zwigato,242767,207267,231336,265971,255191,281840,278139,299522,307042,271010,266425,278637,260341,3445488,5.52,52490,50828,53918,56243,56056,56666,56846,57480,57379,56410,55727,54777,53657,64050,1.42
4,2-WEEKLY,UNKNOWN,385365,271338,305336,428036,387246,461248,448444,484202,503014,441207,433653,469716,438236,5457041,8.74,75163,64603,69936,79665,78436,80453,80687,81159,81490,80108,79557,79330,78074,89399,1.98


In [25]:
cx_data12

,segment,Zwigato_check,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,1-DAILY,Non-Zwigato,81265,70487,78228,90873,86702,97732,96145,101465,103863,92928,91658,94748,87363,1173457,1.88,6827,6787,6977,7092,7087,7142,7127,7122,7108,7108,7051,6960,6856,7185,0.16
1,1-DAILY,UNKNOWN,143507,106138,118190,160005,146149,169047,167437,175550,180569,161992,160483,169408,156570,2015045,3.23,11595,10694,11399,11955,11939,11967,12018,11996,11984,11964,11963,11899,11763,12125,0.27
2,1-DAILY,Zwigato,300676,233558,265897,332994,306196,353971,345809,368293,376274,335497,332657,350646,322251,4224719,6.76,24984,23879,24932,25588,25472,25616,25645,25631,25618,25567,25501,25373,25016,25873,0.57
3,2-WEEKLY,Non-Zwigato,242767,207267,231336,265971,255191,281840,278139,299522,307042,271010,266425,278637,260341,3445488,5.52,52490,50828,53918,56243,56056,56666,56846,57480,57379,56410,55727,54777,53657,64050,1.42
4,2-WEEKLY,UNKNOWN,385365,271338,305336,428036,387246,461248,448444,484202,503014,441207,433653,469716,438236,5457041,8.74,75163,64603,69936,79665,78436,80453,80687,81159,81490,80108,79557,79330,78074,89399,1.98
5,2-WEEKLY,Zwigato,840115,633119,717878,920258,845173,987794,960533,1041171,1072409,942037,935340,998483,927769,11822079,18.93,170847,153318,165238,180617,177851,182406,182586,184593,184627,181315,180930,179259,175858,203844,4.51
6,3-MONTHLY,Non-Zwigato,514458,458765,504357,549648,569018,588528,587566,681705,669957,555401,547607,567185,573466,7367661,11.80,199019,190395,204374,216946,226966,225336,228340,251963,239991,214520,211495,206919,210311,653182,14.44
7,3-MONTHLY,UNKNOWN,383697,277635,310102,386023,368703,440888,443310,513453,517999,438630,432466,468851,461509,5443266,8.71,142105,114025,125361,142891,141605,153867,157243,172560,168483,153237,150101,152089,152551,392318,8.67
8,3-MONTHLY,Zwigato,1028830,815806,906621,1032334,1018910,1155017,1152456,1337033,1338975,1128249,1144438,1213909,1195572,14468150,23.16,393698,339638,370536,400979,403993,424434,431414,473580,457975,417057,419956,418014,420358,1128648,24.95
9,4-OTHER,Non-Zwigato,164871,140220,133739,113695,106070,120745,118981,122806,178146,202342,225652,263981,292482,2183730,3.50,73625,67397,65152,58417,56138,59755,60351,62442,84953,92266,100303,108407,119601,741168,16.38


In [26]:
cx_data12.to_clipboard(index=False)

In [29]:
cx_data03 = f'''
    with base as (
    select 
        customer_id, 
        segment  
    from 
        hive.experiments_internal.customer_segment_amj25
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        count(distinct order_id) orders
    from 
        orders.order_logs_fact
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        -- and order_status = 'dropped'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    ),
    
    appo as (
    
    select
        userid,
        /*
            case 
                when regexp_like(apps_installed, 'jira|miro|zoho|teams||intune|figma|slack|confluence|webex|github') = true 
                then 'office-apps' 
            else 'no-office-apps' end office_tag,  
        */
        CASE 
            WHEN apps_installed LIKE '%uber%' OR apps_installed LIKE '%ola%' OR apps_installed LIKE '%blusmart%' OR apps_installed LIKE '%nammayatri%' OR apps_installed LIKE '%in drive%' 
            OR apps_installed LIKE '%quick ride%' OR apps_installed LIKE '%jugnoo%' 
        THEN 'RHA' 
            else 'NRHA' end as RHA_Check,

        CASE 
            when apps_installed LIKE '%zomato%' OR apps_installed LIKE '%swiggy%' 
        THEN 'Zwigato' 
            else 'Non-Zwigato' end as Zwigato_check

            
    from 
        datasets.customer_apps_installed_snapshot
    )
    
    
    select
        segment,
        
        case 
        when RHA_Check = 'RHA' and Zwigato_check = 'Zwigato' then '1. BOTH'
        when RHA_Check = 'NRHA' and Zwigato_check = 'Zwigato' then '2. ONLY Zwigato'
        when RHA_Check = 'RHA' and Zwigato_check = 'Non-Zwigato' then '3. ONLY RHA'
        when RHA_Check = 'NRHA' and Zwigato_check = 'Non-Zwigato' then '4. NONE'
        ELSE '5. IOS' end tech_savvy,
        
        
        sum(case when week_start_date = '20241216' then orders end ) w01,
        sum(case when week_start_date = '20241223' then orders end ) w02,
        sum(case when week_start_date = '20241230' then orders end ) w03,
        sum(case when week_start_date = '20250106' then orders end ) w04,
        sum(case when week_start_date = '20250113' then orders end ) w05,
        sum(case when week_start_date = '20250120' then orders end ) w06,
        sum(case when week_start_date = '20250127' then orders end ) w07,
        sum(case when week_start_date = '20250203' then orders end ) w08,
        sum(case when week_start_date = '20250210' then orders end ) w09,
        sum(case when week_start_date = '20250217' then orders end ) w10,
        sum(case when week_start_date = '20250224' then orders end ) w11,
        sum(case when week_start_date = '20250303' then orders end ) w12,
        sum(case when week_start_date = '20250310' then orders end ) w13,
        
        sum(orders.orders) as overall,
        100.00*sum(orders)/sum(sum(orders))over() as pct,
        
        count(distinct case when week_start_date = '20241216' then orders.customer_id end ) C_w01,
        count(distinct case when week_start_date = '20241223' then orders.customer_id end ) C_w02,
        count(distinct case when week_start_date = '20241230' then orders.customer_id end ) C_w03,
        count(distinct case when week_start_date = '20250106' then orders.customer_id end ) C_w04,
        count(distinct case when week_start_date = '20250113' then orders.customer_id end ) C_w05,
        count(distinct case when week_start_date = '20250120' then orders.customer_id end ) C_w06,
        count(distinct case when week_start_date = '20250127' then orders.customer_id end ) C_w07,
        count(distinct case when week_start_date = '20250203' then orders.customer_id end ) C_w08,
        count(distinct case when week_start_date = '20250210' then orders.customer_id end ) C_w09,
        count(distinct case when week_start_date = '20250217' then orders.customer_id end ) C_w10,
        count(distinct case when week_start_date = '20250224' then orders.customer_id end ) C_w11,
        count(distinct case when week_start_date = '20250303' then orders.customer_id end ) C_w12,
        count(distinct case when week_start_date = '20250310' then orders.customer_id end ) C_w13,
        
        count(distinct orders.customer_id) as C_overall,
        100.00*count(distinct orders.customer_id)/sum(count(distinct orders.customer_id))over() as c_pct
      
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    left join
        appo
        on base.customer_id = appo.userid
    group by 1,2
    order by 1,2
'''
cx_data13 = pd.read_sql(cx_data03, conn1)
cx_data13.head()

,segment,tech_savvy,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,1-DAILY,1. BOTH,270026,207568,236618,299104,273606,318113,310511,330396,337475,300427,297987,314301,288772,3784904,6.06,22378,21300,22276,22906,22793,22930,22952,22945,22927,22879,22831,22711,22397,23155,0.51
1,1-DAILY,2. ONLY Zwigato,30650,25990,29279,33890,32590,35858,35298,37897,38799,35070,34670,36345,33479,439815,0.70,2606,2579,2656,2682,2679,2686,2693,2686,2691,2688,2670,2662,2619,2718,0.06
2,1-DAILY,3. ONLY RHA,49119,41870,46372,53994,51071,58149,57466,60384,60977,55182,54446,56387,51819,697236,1.12,4064,4019,4133,4205,4198,4229,4222,4218,4205,4213,4182,4137,4080,4253,0.09
3,1-DAILY,4. NONE,32146,28617,31856,36879,35631,39583,38679,41081,42886,37746,37212,38361,35544,476221,0.76,2763,2768,2844,2887,2889,2913,2905,2904,2903,2895,2869,2823,2776,2932,0.06
4,1-DAILY,5. IOS,143507,106138,118190,160005,146149,169047,167437,175550,180569,161992,160483,169408,156570,2015045,3.23,11595,10694,11399,11955,11939,11967,12018,11996,11984,11964,11963,11899,11763,12125,0.27


In [30]:
cx_data13

,segment,tech_savvy,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,1-DAILY,1. BOTH,270026,207568,236618,299104,273606,318113,310511,330396,337475,300427,297987,314301,288772,3784904,6.06,22378,21300,22276,22906,22793,22930,22952,22945,22927,22879,22831,22711,22397,23155,0.51
1,1-DAILY,2. ONLY Zwigato,30650,25990,29279,33890,32590,35858,35298,37897,38799,35070,34670,36345,33479,439815,0.70,2606,2579,2656,2682,2679,2686,2693,2686,2691,2688,2670,2662,2619,2718,0.06
2,1-DAILY,3. ONLY RHA,49119,41870,46372,53994,51071,58149,57466,60384,60977,55182,54446,56387,51819,697236,1.12,4064,4019,4133,4205,4198,4229,4222,4218,4205,4213,4182,4137,4080,4253,0.09
3,1-DAILY,4. NONE,32146,28617,31856,36879,35631,39583,38679,41081,42886,37746,37212,38361,35544,476221,0.76,2763,2768,2844,2887,2889,2913,2905,2904,2903,2895,2869,2823,2776,2932,0.06
4,1-DAILY,5. IOS,143507,106138,118190,160005,146149,169047,167437,175550,180569,161992,160483,169408,156570,2015045,3.23,11595,10694,11399,11955,11939,11967,12018,11996,11984,11964,11963,11899,11763,12125,0.27
5,2-WEEKLY,1. BOTH,748611,554623,628987,819109,749050,881212,855496,926768,956614,839341,833976,892082,826344,10512213,16.83,151087,134338,145100,159567,157079,161230,161319,163089,163143,160269,159955,158513,155449,179867,3.98
6,2-WEEKLY,2. ONLY Zwigato,91504,78496,88891,101149,96123,106582,105037,114403,115795,102696,101364,106401,101425,1309866,2.10,19760,18980,20138,21050,20772,21176,21267,21504,21484,21046,20975,20746,20409,23977,0.53
7,2-WEEKLY,3. ONLY RHA,136650,114332,127308,148400,141032,157702,155735,168175,171056,151836,148167,155480,145601,1921474,3.08,29449,28081,29957,31385,31202,31727,31787,32118,32038,31497,31105,30692,30107,35684,0.79
8,2-WEEKLY,4. NONE,106117,92935,104028,117571,114159,124138,122404,131347,135986,119174,118258,123157,114740,1524014,2.44,23041,22747,23961,24858,24854,24939,25059,25362,25341,24913,24622,24085,23550,28366,0.63
9,2-WEEKLY,5. IOS,385365,271338,305336,428036,387246,461248,448444,484202,503014,441207,433653,469716,438236,5457041,8.74,75163,64603,69936,79665,78436,80453,80687,81159,81490,80108,79557,79330,78074,89399,1.98


In [34]:
cx_data13.to_clipboard(index=False)

In [31]:
cx_data04 = f'''
    with base as (
    select 
        customer_id, 
        segment  
    from 
        hive.experiments_internal.customer_segment_amj25
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        count(distinct order_id) orders
    from 
        orders.order_logs_fact
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        -- and order_status = 'dropped'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    ),
    
    appo as (
    
    select
        userid,
        /*
            case 
                when regexp_like(apps_installed, 'jira|miro|zoho|teams||intune|figma|slack|confluence|webex|github') = true 
                then 'office-apps' 
            else 'no-office-apps' end office_tag,  
        */
        CASE 
            WHEN apps_installed LIKE '%uber%' OR apps_installed LIKE '%ola%' OR apps_installed LIKE '%blusmart%' OR apps_installed LIKE '%nammayatri%' OR apps_installed LIKE '%in drive%' 
            OR apps_installed LIKE '%quick ride%' OR apps_installed LIKE '%jugnoo%' 
        THEN 'RHA' 
            else 'NRHA' end as RHA_Check,

        CASE 
            when apps_installed LIKE '%zomato%' OR apps_installed LIKE '%swiggy%' 
        THEN 'Zwigato' 
            else 'Non-Zwigato' end as Zwigato_check

            
    from 
        datasets.customer_apps_installed_snapshot
    )
    
    
    select
        -- segment,
        
        case 
        when RHA_Check = 'RHA' and Zwigato_check = 'Zwigato' then '1. BOTH'
        when RHA_Check = 'NRHA' and Zwigato_check = 'Zwigato' then '2. ONLY Zwigato'
        when RHA_Check = 'RHA' and Zwigato_check = 'Non-Zwigato' then '3. ONLY RHA'
        when RHA_Check = 'NRHA' and Zwigato_check = 'Non-Zwigato' then '4. NONE'
        ELSE '5. IOS' end tech_savvy,
        
        
        sum(case when week_start_date = '20241216' then orders end ) w01,
        sum(case when week_start_date = '20241223' then orders end ) w02,
        sum(case when week_start_date = '20241230' then orders end ) w03,
        sum(case when week_start_date = '20250106' then orders end ) w04,
        sum(case when week_start_date = '20250113' then orders end ) w05,
        sum(case when week_start_date = '20250120' then orders end ) w06,
        sum(case when week_start_date = '20250127' then orders end ) w07,
        sum(case when week_start_date = '20250203' then orders end ) w08,
        sum(case when week_start_date = '20250210' then orders end ) w09,
        sum(case when week_start_date = '20250217' then orders end ) w10,
        sum(case when week_start_date = '20250224' then orders end ) w11,
        sum(case when week_start_date = '20250303' then orders end ) w12,
        sum(case when week_start_date = '20250310' then orders end ) w13,
        
        sum(orders.orders) as overall,
        100.00*sum(orders)/sum(sum(orders))over() as pct,
        
        count(distinct case when week_start_date = '20241216' then orders.customer_id end ) C_w01,
        count(distinct case when week_start_date = '20241223' then orders.customer_id end ) C_w02,
        count(distinct case when week_start_date = '20241230' then orders.customer_id end ) C_w03,
        count(distinct case when week_start_date = '20250106' then orders.customer_id end ) C_w04,
        count(distinct case when week_start_date = '20250113' then orders.customer_id end ) C_w05,
        count(distinct case when week_start_date = '20250120' then orders.customer_id end ) C_w06,
        count(distinct case when week_start_date = '20250127' then orders.customer_id end ) C_w07,
        count(distinct case when week_start_date = '20250203' then orders.customer_id end ) C_w08,
        count(distinct case when week_start_date = '20250210' then orders.customer_id end ) C_w09,
        count(distinct case when week_start_date = '20250217' then orders.customer_id end ) C_w10,
        count(distinct case when week_start_date = '20250224' then orders.customer_id end ) C_w11,
        count(distinct case when week_start_date = '20250303' then orders.customer_id end ) C_w12,
        count(distinct case when week_start_date = '20250310' then orders.customer_id end ) C_w13,
        
        count(distinct orders.customer_id) as C_overall,
        100.00*count(distinct orders.customer_id)/sum(count(distinct orders.customer_id))over() as c_pct
      
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    left join
        appo
        on base.customer_id = appo.userid
    group by 1
    order by 1
'''
cx_data14 = pd.read_sql(cx_data04, conn1)
cx_data14.head()

,tech_savvy,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,1. BOTH,2089508,1596573,1773213,2127212,2004649,2336802,2300145,2555249,2663453,2350167,2384250,2573395,2480108,29234724,46.80,589313,508535,547226,581604,577275,609134,616555,655795,668925,636561,647295,654380,660448,1818974,40.20
1,2. ONLY Zwigato,334075,287464,322906,345008,336314,362794,361033,408357,423040,377346,386267,413664,414202,4772470,7.64,105921,99091,106688,108910,108467,109812,111591,121023,123300,117965,121319,122611,126085,414674,9.17
2,3. ONLY RHA,519226,448555,480751,520748,513108,562927,558988,620358,640665,573468,574185,612228,606467,7231674,11.58,169101,158481,165152,168572,171407,176187,178282,190401,194053,184754,185767,186688,190871,680685,15.05
3,4. NONE,509520,451489,492232,524616,529118,552643,549194,615407,649575,576709,586101,623283,639311,7299198,11.69,173359,167030,176115,180086,184868,182760,185121,200002,207004,196587,200047,201976,211775,841090,18.59
4,5. IOS,983178,709725,788424,1020053,943003,1123674,1111789,1227622,1285929,1141959,1138595,1246308,1205478,13925737,22.29,257924,213076,230957,255544,251419,269101,273359,290098,297016,284270,283771,290091,293247,768832,16.99


In [32]:
cx_data14

,tech_savvy,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,1. BOTH,2089508,1596573,1773213,2127212,2004649,2336802,2300145,2555249,2663453,2350167,2384250,2573395,2480108,29234724,46.80,589313,508535,547226,581604,577275,609134,616555,655795,668925,636561,647295,654380,660448,1818974,40.20
1,2. ONLY Zwigato,334075,287464,322906,345008,336314,362794,361033,408357,423040,377346,386267,413664,414202,4772470,7.64,105921,99091,106688,108910,108467,109812,111591,121023,123300,117965,121319,122611,126085,414674,9.17
2,3. ONLY RHA,519226,448555,480751,520748,513108,562927,558988,620358,640665,573468,574185,612228,606467,7231674,11.58,169101,158481,165152,168572,171407,176187,178282,190401,194053,184754,185767,186688,190871,680685,15.05
3,4. NONE,509520,451489,492232,524616,529118,552643,549194,615407,649575,576709,586101,623283,639311,7299198,11.69,173359,167030,176115,180086,184868,182760,185121,200002,207004,196587,200047,201976,211775,841090,18.59
4,5. IOS,983178,709725,788424,1020053,943003,1123674,1111789,1227622,1285929,1141959,1138595,1246308,1205478,13925737,22.29,257924,213076,230957,255544,251419,269101,273359,290098,297016,284270,283771,290091,293247,768832,16.99


In [33]:
cx_data14.to_clipboard(index=False)

### regularity_segment

In [64]:
cx_data = f'''
    with base as (
    select 
        customer_id,
        taxi_regularity_segment
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        count(distinct order_id) orders
    from 
        orders.order_logs_fact
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        and order_status='dropped'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    )
    
    
    select
        taxi_regularity_segment regularity_segment,
        sum(case when week_start_date = '20241216' then orders end ) w01,
        sum(case when week_start_date = '20241223' then orders end ) w02,
        sum(case when week_start_date = '20241230' then orders end ) w03,
        sum(case when week_start_date = '20250106' then orders end ) w04,
        sum(case when week_start_date = '20250113' then orders end ) w05,
        sum(case when week_start_date = '20250120' then orders end ) w06,
        sum(case when week_start_date = '20250127' then orders end ) w07,
        sum(case when week_start_date = '20250203' then orders end ) w08,
        sum(case when week_start_date = '20250210' then orders end ) w09,
        sum(case when week_start_date = '20250217' then orders end ) w10,
        sum(case when week_start_date = '20250224' then orders end ) w11,
        sum(case when week_start_date = '20250303' then orders end ) w12,
        sum(case when week_start_date = '20250310' then orders end ) w13,
        
        sum(orders.orders) as overall,
        100.00*sum(orders)/sum(sum(orders))over() as pct,
        
        count(distinct case when week_start_date = '20241216' then orders.customer_id end ) C_w01,
        count(distinct case when week_start_date = '20241223' then orders.customer_id end ) C_w02,
        count(distinct case when week_start_date = '20241230' then orders.customer_id end ) C_w03,
        count(distinct case when week_start_date = '20250106' then orders.customer_id end ) C_w04,
        count(distinct case when week_start_date = '20250113' then orders.customer_id end ) C_w05,
        count(distinct case when week_start_date = '20250120' then orders.customer_id end ) C_w06,
        count(distinct case when week_start_date = '20250127' then orders.customer_id end ) C_w07,
        count(distinct case when week_start_date = '20250203' then orders.customer_id end ) C_w08,
        count(distinct case when week_start_date = '20250210' then orders.customer_id end ) C_w09,
        count(distinct case when week_start_date = '20250217' then orders.customer_id end ) C_w10,
        count(distinct case when week_start_date = '20250224' then orders.customer_id end ) C_w11,
        count(distinct case when week_start_date = '20250303' then orders.customer_id end ) C_w12,
        count(distinct case when week_start_date = '20250310' then orders.customer_id end ) C_w13,
        
        count(distinct orders.customer_id) as C_overall,
        100.00*count(distinct orders.customer_id)/sum(count(distinct orders.customer_id))over() as c_pct

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1
       
'''
cx_data1 = pd.read_sql(cx_data, conn3)
cx_data1.head()

,regularity_segment,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,UNKNOWN,85173,82210,86270,87752,92075,96283,102358,118504,128483,127044,133003,142128,167341,1448624,3.69,70273,67554,71390,72382,75964,78992,84057,96624,104437,102609,107688,113425,129072,904251,20.99
1,BI_WEEKLY,463836,428413,456514,465139,459308,470027,475889,512661,528376,526807,553734,577747,594309,6512760,16.59,243251,227117,244438,251212,253081,260094,267023,290251,299402,292542,303011,303466,309126,898962,20.87
2,RARE_NEED,271456,199399,177741,154690,135576,129530,130498,136368,137473,133105,132584,133210,154526,2026156,5.16,151105,114629,103547,87940,79703,74048,73798,75969,77076,72734,73817,72091,80235,732184,17.00
3,MONTHLY,237761,229851,237309,231071,226627,222571,219630,228962,221747,202208,195420,218094,245168,2916419,7.43,134300,130500,140406,142405,140631,139688,137397,143278,140282,127439,124934,132325,138442,802848,18.64
4,WEEKLY,794420,705913,781152,849783,838963,902318,933667,988031,1001641,974002,975882,995510,1014385,11755667,29.94,350426,319928,350752,385914,387910,409639,420881,441532,448117,439364,441295,441258,446135,710546,16.50


In [65]:
cx_data1

,regularity_segment,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,UNKNOWN,85173,82210,86270,87752,92075,96283,102358,118504,128483,127044,133003,142128,167341,1448624,3.69,70273,67554,71390,72382,75964,78992,84057,96624,104437,102609,107688,113425,129072,904251,20.99
1,BI_WEEKLY,463836,428413,456514,465139,459308,470027,475889,512661,528376,526807,553734,577747,594309,6512760,16.59,243251,227117,244438,251212,253081,260094,267023,290251,299402,292542,303011,303466,309126,898962,20.87
2,RARE_NEED,271456,199399,177741,154690,135576,129530,130498,136368,137473,133105,132584,133210,154526,2026156,5.16,151105,114629,103547,87940,79703,74048,73798,75969,77076,72734,73817,72091,80235,732184,17.00
3,MONTHLY,237761,229851,237309,231071,226627,222571,219630,228962,221747,202208,195420,218094,245168,2916419,7.43,134300,130500,140406,142405,140631,139688,137397,143278,140282,127439,124934,132325,138442,802848,18.64
4,WEEKLY,794420,705913,781152,849783,838963,902318,933667,988031,1001641,974002,975882,995510,1014385,11755667,29.94,350426,319928,350752,385914,387910,409639,420881,441532,448117,439364,441295,441258,446135,710546,16.50
5,DAILY,989760,810897,906525,1098514,1050477,1169596,1218140,1243655,1250408,1240640,1209059,1229595,1184923,14602189,37.19,186668,165715,179513,202252,204496,214498,219241,224151,226495,224028,221899,220577,217315,258297,6.00


In [66]:
cx_data1.to_clipboard()

### ps_tag_link

In [17]:
cx_data2 = f'''
    with base as (
    select 
        customer_id,
        ps_tag_link
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        count(distinct order_id) orders
    from 
        orders.order_logs_fact
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        and service_obj_service_name in ('Link','Bike Lite','Bike Pink','Bike Metro')
        -- and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    )
    
    select
        ps_tag_link ps_tag_link,
        sum(case when week_start_date = '20241216' then orders end ) w01,
        sum(case when week_start_date = '20241223' then orders end ) w02,
        sum(case when week_start_date = '20241230' then orders end ) w03,
        sum(case when week_start_date = '20250106' then orders end ) w04,
        sum(case when week_start_date = '20250113' then orders end ) w05,
        sum(case when week_start_date = '20250120' then orders end ) w06,
        sum(case when week_start_date = '20250127' then orders end ) w07,
        sum(case when week_start_date = '20250203' then orders end ) w08,
        sum(case when week_start_date = '20250210' then orders end ) w09,
        sum(case when week_start_date = '20250217' then orders end ) w10,
        sum(case when week_start_date = '20250224' then orders end ) w11,
        sum(case when week_start_date = '20250303' then orders end ) w12,
        sum(case when week_start_date = '20250310' then orders end ) w13,

        sum(orders.orders) as overall,
        100.00*sum(orders)/sum(sum(orders))over() as pct,

        count(distinct orders.customer_id) as C_overall,
        100.00*count(distinct orders.customer_id)/sum(count(distinct orders.customer_id))over() as c_pct
                
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1
       
'''
cx_data2 = pd.read_sql(cx_data2, conn4)
cx_data2.head()

,ps_tag_link,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_overall,c_pct
0,UNKNOWN,873169,643244,725046,842731,807476,922411,913131,1011590,1051622,943119,988484,1102151,1076411,11900585,61.70,2328110,86.95
1,PS,165489,126724,149801,190820,174862,195722,189447,209405,219719,184131,192356,214961,199503,2412940,12.51,139399,5.21
2,NPS,336099,269741,321300,418071,370675,412261,394780,439058,444862,368949,383321,423750,390857,4973724,25.79,209946,7.84


In [18]:
cx_data2

,ps_tag_link,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_overall,c_pct
0,UNKNOWN,873169,643244,725046,842731,807476,922411,913131,1011590,1051622,943119,988484,1102151,1076411,11900585,61.70,2328110,86.95
1,PS,165489,126724,149801,190820,174862,195722,189447,209405,219719,184131,192356,214961,199503,2412940,12.51,139399,5.21
2,NPS,336099,269741,321300,418071,370675,412261,394780,439058,444862,368949,383321,423750,390857,4973724,25.79,209946,7.84


In [18]:
cx_data2.to_clipboard(index=False)

### ps_tag_auto

In [19]:
cx_data3 = f'''
    with base as (
    select 
        customer_id,
        ps_tag_auto
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        count(distinct order_id) orders
    from 
        orders.order_logs_fact
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium')
        -- and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    )
    
    
    select
        ps_tag_auto ps_tag_auto,
        sum(case when week_start_date = '20241216' then orders end ) w01,
        sum(case when week_start_date = '20241223' then orders end ) w02,
        sum(case when week_start_date = '20241230' then orders end ) w03,
        sum(case when week_start_date = '20250106' then orders end ) w04,
        sum(case when week_start_date = '20250113' then orders end ) w05,
        sum(case when week_start_date = '20250120' then orders end ) w06,
        sum(case when week_start_date = '20250127' then orders end ) w07,
        sum(case when week_start_date = '20250203' then orders end ) w08,
        sum(case when week_start_date = '20250210' then orders end ) w09,
        sum(case when week_start_date = '20250217' then orders end ) w10,
        sum(case when week_start_date = '20250224' then orders end ) w11,
        sum(case when week_start_date = '20250303' then orders end ) w12,
        sum(case when week_start_date = '20250310' then orders end ) w13,

        sum(orders.orders) as overall,
        100.00*sum(orders)/sum(sum(orders))over() as pct,

        count(distinct orders.customer_id) as C_overall,
        100.00*count(distinct orders.customer_id)/sum(count(distinct orders.customer_id))over() as c_pct

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1
       
'''
cx_data3 = pd.read_sql(cx_data3, conn4)
cx_data3.head()

,ps_tag_auto,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_overall,c_pct
0,UNKNOWN,544724,384027,422817,511972,510093,689510,791900,951080,1030599,952683,976850,1072176,1107027,9945458,27.69,2128906,61.38
1,PS,700261,604129,643209,718729,682567,716753,661643,715371,725781,644904,635801,671487,651494,8772129,24.42,533148,15.37
2,NPS,1298731,992695,1085258,1353100,1251278,1433952,1365166,1478852,1513953,1344399,1328742,1407436,1349822,17203384,47.89,806080,23.24


In [25]:
cx_data3

,ps_tag_auto,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_overall,c_pct
0,UNKNOWN,544724,384027,422817,511972,510093,689510,791900,951080,1030599,952683,976850,1072176,1107027,9945458,27.69,2128906,61.38
1,PS,700261,604129,643209,718729,682567,716753,661643,715371,725781,644904,635801,671487,651494,8772129,24.42,533148,15.37
2,NPS,1298731,992695,1085258,1353100,1251278,1433952,1365166,1478852,1513953,1344399,1328742,1407436,1349822,17203384,47.89,806080,23.24


In [13]:
cx_data3.to_clipboard(index=False)

### taxi_income_segment

In [41]:
cx_data4 = f'''
    with base as (
    select 
        customer_id,
        taxi_income_segment
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        count(distinct order_id) orders
    from 
        orders.order_logs_fact
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        and order_status = 'dropped'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    )
    
    
    select
        taxi_income_segment taxi_income_segment,
        sum(case when week_start_date = '20241216' then orders end ) w01,
        sum(case when week_start_date = '20241223' then orders end ) w02,
        sum(case when week_start_date = '20241230' then orders end ) w03,
        sum(case when week_start_date = '20250106' then orders end ) w04,
        sum(case when week_start_date = '20250113' then orders end ) w05,
        sum(case when week_start_date = '20250120' then orders end ) w06,
        sum(case when week_start_date = '20250127' then orders end ) w07,
        sum(case when week_start_date = '20250203' then orders end ) w08,
        sum(case when week_start_date = '20250210' then orders end ) w09,
        sum(case when week_start_date = '20250217' then orders end ) w10,
        sum(case when week_start_date = '20250224' then orders end ) w11,
        sum(case when week_start_date = '20250303' then orders end ) w12,
        sum(case when week_start_date = '20250310' then orders end ) w13,

        sum(orders.orders) as overall,
        100.00*sum(orders)/sum(sum(orders))over() as pct,
        
        count(distinct case when week_start_date = '20241216' then orders.customer_id end ) C_w01,
        count(distinct case when week_start_date = '20241223' then orders.customer_id end ) C_w02,
        count(distinct case when week_start_date = '20241230' then orders.customer_id end ) C_w03,
        count(distinct case when week_start_date = '20250106' then orders.customer_id end ) C_w04,
        count(distinct case when week_start_date = '20250113' then orders.customer_id end ) C_w05,
        count(distinct case when week_start_date = '20250120' then orders.customer_id end ) C_w06,
        count(distinct case when week_start_date = '20250127' then orders.customer_id end ) C_w07,
        count(distinct case when week_start_date = '20250203' then orders.customer_id end ) C_w08,
        count(distinct case when week_start_date = '20250210' then orders.customer_id end ) C_w09,
        count(distinct case when week_start_date = '20250217' then orders.customer_id end ) C_w10,
        count(distinct case when week_start_date = '20250224' then orders.customer_id end ) C_w11,
        count(distinct case when week_start_date = '20250303' then orders.customer_id end ) C_w12,
        count(distinct case when week_start_date = '20250310' then orders.customer_id end ) C_w13,
        
        count(distinct orders.customer_id) as C_overall,
        100.00*count(distinct orders.customer_id)/sum(count(distinct orders.customer_id))over() as c_pct
      
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1
       
'''
cx_data4 = pd.read_sql(cx_data4, conn1)
cx_data4.head()

,taxi_income_segment,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,HIGH_INCOME,1425044,1211912,1310382,1445645,1393608,1502188,1548466,1612807,1628498,1599188,1592357,1639872,1658363,19568330,49.84,557612,497647,530658,560561,555820,579207,591795,622955,632701,615196,620634,625005,637659,1997083,46.37
1,MEDIUM_INCOME,643410,584722,619908,653558,640742,669396,683737,724357,732508,711756,713292,732662,752666,8862714,22.57,272469,257328,269795,276667,279463,282116,286669,304688,310306,299211,303494,304441,313926,1136600,26.39
2,UNKNOWN,662949,555967,606122,673657,655514,704905,732120,766911,779460,770500,772473,800198,819041,9299817,23.69,258347,224441,241612,255992,256541,266641,274315,291023,297945,292094,295936,301439,313239,957009,22.22
3,LOW_INCOME,111003,104082,109099,114089,113162,113836,115859,124106,127662,122362,121560,123552,130582,1530954,3.90,47595,46027,47981,48885,49961,48995,49618,53139,54857,52215,52580,52257,55501,216396,5.02


In [57]:
cx_data4

,taxi_income_segment,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,HIGH_INCOME,1425044,1211912,1310382,1445645,1393608,1502188,1548466,1612807,1628498,1599188,1592357,1639872,1658363,19568330,49.84,557612,497647,530658,560561,555820,579207,591795,622955,632701,615196,620634,625005,637659,1997083,46.37
1,MEDIUM_INCOME,643410,584722,619908,653558,640742,669396,683737,724357,732508,711756,713292,732662,752666,8862714,22.57,272469,257328,269795,276667,279463,282116,286669,304688,310306,299211,303494,304441,313926,1136600,26.39
2,UNKNOWN,662949,555967,606122,673657,655514,704905,732120,766911,779460,770500,772473,800198,819041,9299817,23.69,258347,224441,241612,255992,256541,266641,274315,291023,297945,292094,295936,301439,313239,957009,22.22
3,LOW_INCOME,111003,104082,109099,114089,113162,113836,115859,124106,127662,122362,121560,123552,130582,1530954,3.90,47595,46027,47981,48885,49961,48995,49618,53139,54857,52215,52580,52257,55501,216396,5.02


In [17]:
cx_data4.to_clipboard(index=False)

### taxi_service_affinity

In [42]:
cx_data5 = f'''
    with base as (
    select 
        customer_id,
        taxi_service_affinity
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        count(distinct order_id) orders
    from 
        orders.order_logs_fact
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        and order_status = 'dropped'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    )
    
    
    select
        taxi_service_affinity taxi_service_affinity,
        sum(case when week_start_date = '20241216' then orders end ) w01,
        sum(case when week_start_date = '20241223' then orders end ) w02,
        sum(case when week_start_date = '20241230' then orders end ) w03,
        sum(case when week_start_date = '20250106' then orders end ) w04,
        sum(case when week_start_date = '20250113' then orders end ) w05,
        sum(case when week_start_date = '20250120' then orders end ) w06,
        sum(case when week_start_date = '20250127' then orders end ) w07,
        sum(case when week_start_date = '20250203' then orders end ) w08,
        sum(case when week_start_date = '20250210' then orders end ) w09,
        sum(case when week_start_date = '20250217' then orders end ) w10,
        sum(case when week_start_date = '20250224' then orders end ) w11,
        sum(case when week_start_date = '20250303' then orders end ) w12,
        sum(case when week_start_date = '20250310' then orders end ) w13,

        sum(orders.orders) as overall,
        100.00*sum(orders)/sum(sum(orders))over() as pct,
        
        count(distinct case when week_start_date = '20241216' then orders.customer_id end ) C_w01,
        count(distinct case when week_start_date = '20241223' then orders.customer_id end ) C_w02,
        count(distinct case when week_start_date = '20241230' then orders.customer_id end ) C_w03,
        count(distinct case when week_start_date = '20250106' then orders.customer_id end ) C_w04,
        count(distinct case when week_start_date = '20250113' then orders.customer_id end ) C_w05,
        count(distinct case when week_start_date = '20250120' then orders.customer_id end ) C_w06,
        count(distinct case when week_start_date = '20250127' then orders.customer_id end ) C_w07,
        count(distinct case when week_start_date = '20250203' then orders.customer_id end ) C_w08,
        count(distinct case when week_start_date = '20250210' then orders.customer_id end ) C_w09,
        count(distinct case when week_start_date = '20250217' then orders.customer_id end ) C_w10,
        count(distinct case when week_start_date = '20250224' then orders.customer_id end ) C_w11,
        count(distinct case when week_start_date = '20250303' then orders.customer_id end ) C_w12,
        count(distinct case when week_start_date = '20250310' then orders.customer_id end ) C_w13,
        
        count(distinct orders.customer_id) as C_overall,
        100.00*count(distinct orders.customer_id)/sum(count(distinct orders.customer_id))over() as c_pct
      
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1
       
'''
cx_data5 = pd.read_sql(cx_data5, conn1)
cx_data5.head()

,taxi_service_affinity,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,NO_AFFINITY,271561,254934,278523,293430,295140,308562,321023,333814,336815,328107,329027,330103,331313,4012352,10.22,83900,79363,85444,89766,90830,93410,95552,98459,99609,97074,97112,95943,95525,144935,3.37
1,AUTO_CAB,63088,56466,59397,65981,65734,70139,71919,74234,75495,73705,71965,74294,74044,896461,2.28,22689,20700,21968,23640,24094,24881,25434,26192,26817,25957,25843,25860,25650,46468,1.08
2,LINK_CAB,3975,3927,4276,4530,4610,4686,4766,5280,5519,5499,5348,5462,5680,63558,0.16,2170,2107,2271,2412,2460,2516,2586,2737,2863,2787,2764,2730,2751,6231,0.14
3,ONLY_CAB,76698,69221,74654,88489,86182,94549,95011,96028,97243,93824,90814,93927,93525,1150165,2.93,27525,25644,27742,30669,30979,32599,32794,33669,34405,33303,32905,32755,32511,65631,1.52
4,ONLY_AUTO,1191479,984155,1074816,1240334,1183967,1290145,1324068,1365931,1374603,1355696,1333480,1381611,1361871,16462156,41.93,349352,311827,334674,366354,365978,380960,386696,399961,403183,394195,393177,392724,390791,640049,14.86


In [61]:
cx_data5

,taxi_service_affinity,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,NO_AFFINITY,271561,254934,278523,293430,295140,308562,321023,333814,336815,328107,329027,330103,331313,4012352,10.22,83900,79363,85444,89766,90830,93410,95552,98459,99609,97074,97112,95943,95525,144935,3.37
1,AUTO_CAB,63088,56466,59397,65981,65734,70139,71919,74234,75495,73705,71965,74294,74044,896461,2.28,22689,20700,21968,23640,24094,24881,25434,26192,26817,25957,25843,25860,25650,46468,1.08
2,LINK_CAB,3975,3927,4276,4530,4610,4686,4766,5280,5519,5499,5348,5462,5680,63558,0.16,2170,2107,2271,2412,2460,2516,2586,2737,2863,2787,2764,2730,2751,6231,0.14
3,ONLY_CAB,76698,69221,74654,88489,86182,94549,95011,96028,97243,93824,90814,93927,93525,1150165,2.93,27525,25644,27742,30669,30979,32599,32794,33669,34405,33303,32905,32755,32511,65631,1.52
4,ONLY_AUTO,1191479,984155,1074816,1240334,1183967,1290145,1324068,1365931,1374603,1355696,1333480,1381611,1361871,16462156,41.93,349352,311827,334674,366354,365978,380960,386696,399961,403183,394195,393177,392724,390791,640049,14.86
5,UNKNOWN,695658,600209,620590,607277,605824,616808,637298,702726,724616,706126,725485,758672,846530,8847819,22.54,471914,418717,436772,433700,433134,440076,452621,496260,512308,494529,508878,522576,564709,3033495,70.43
6,ONLY_LINK,425676,384096,419305,465936,443220,479210,496188,513029,514475,505866,507265,512413,506344,6173023,15.72,135555,126975,137824,149601,148166,154663,158014,163459,164961,160808,161626,160443,158231,282104,6.55
7,LINK_AUTO,114271,103675,113950,120972,118349,126226,129909,137139,139362,134983,136298,139802,141345,1656281,4.22,42918,40110,43351,45963,46144,47854,48700,51068,51663,50063,50339,50111,50157,88175,2.05


### RHA Check

In [43]:
cx_data6 = f'''
with base as (
    select 
        customer_id,
        taxi_regularity_segment
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        count(distinct order_id) orders
    from 
        orders.order_logs_fact
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        and order_status = 'dropped'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    ),
    
    appo as (
    
    select
        userid,
            case 
                when regexp_like(apps_installed, 'jira|miro|zoho|teams||intune|figma|slack|confluence|webex|github') = true 
                then 'office-apps' 
            else 'no-office-apps' end office_tag,  
        
        CASE 
            WHEN apps_installed LIKE '%uber%' OR apps_installed LIKE '%ola%' OR apps_installed LIKE '%blusmart%' OR apps_installed LIKE '%nammayatri%' OR apps_installed LIKE '%in drive%' 
            OR apps_installed LIKE '%quick ride%' OR apps_installed LIKE '%jugnoo%' 
        THEN 'RHA' 
            else 'NRHA' end as RHA_Check,

        CASE 
            when apps_installed LIKE '%zomato%' OR apps_installed LIKE '%swiggy%' 
        THEN 'Zwigato' 
            else 'Non-Zwigato' end as Zwigato_check

            
    from 
        datasets.customer_apps_installed_snapshot
    )
    
    
    select
        coalesce(RHA_Check, 'UNKNOWN') RHA_Check,
        sum(case when week_start_date = '20241216' then orders end ) w01,
        sum(case when week_start_date = '20241223' then orders end ) w02,
        sum(case when week_start_date = '20241230' then orders end ) w03,
        sum(case when week_start_date = '20250106' then orders end ) w04,
        sum(case when week_start_date = '20250113' then orders end ) w05,
        sum(case when week_start_date = '20250120' then orders end ) w06,
        sum(case when week_start_date = '20250127' then orders end ) w07,
        sum(case when week_start_date = '20250203' then orders end ) w08,
        sum(case when week_start_date = '20250210' then orders end ) w09,
        sum(case when week_start_date = '20250217' then orders end ) w10,
        sum(case when week_start_date = '20250224' then orders end ) w11,
        sum(case when week_start_date = '20250303' then orders end ) w12,
        sum(case when week_start_date = '20250310' then orders end ) w13,
        
        sum(orders.orders) as overall,
        100.00*sum(orders)/sum(sum(orders))over() as pct,
        
        count(distinct case when week_start_date = '20241216' then orders.customer_id end ) C_w01,
        count(distinct case when week_start_date = '20241223' then orders.customer_id end ) C_w02,
        count(distinct case when week_start_date = '20241230' then orders.customer_id end ) C_w03,
        count(distinct case when week_start_date = '20250106' then orders.customer_id end ) C_w04,
        count(distinct case when week_start_date = '20250113' then orders.customer_id end ) C_w05,
        count(distinct case when week_start_date = '20250120' then orders.customer_id end ) C_w06,
        count(distinct case when week_start_date = '20250127' then orders.customer_id end ) C_w07,
        count(distinct case when week_start_date = '20250203' then orders.customer_id end ) C_w08,
        count(distinct case when week_start_date = '20250210' then orders.customer_id end ) C_w09,
        count(distinct case when week_start_date = '20250217' then orders.customer_id end ) C_w10,
        count(distinct case when week_start_date = '20250224' then orders.customer_id end ) C_w11,
        count(distinct case when week_start_date = '20250303' then orders.customer_id end ) C_w12,
        count(distinct case when week_start_date = '20250310' then orders.customer_id end ) C_w13,
        
        count(distinct orders.customer_id) as C_overall,
        100.00*count(distinct orders.customer_id)/sum(count(distinct orders.customer_id))over() as c_pct
      
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    left join
        appo
        on base.customer_id = appo.userid
    group by 1
    order by 1 
'''
cx_data6 = pd.read_sql(cx_data6, conn1)
cx_data6.head()

,RHA_Check,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,NRHA,548623,522559,555232,573336,568018,574641,589878,631868,647282,630896,634523,654946,691053,7822855,19.92,249313,241499,255155,260351,263665,260874,266203,286179,294155,284751,290407,293174,309117,1203396,27.94
1,RHA,1665048,1433183,1541684,1672499,1621501,1745114,1793116,1876233,1892782,1853671,1853702,1903355,1928712,22780600,58.02,661318,594251,630263,656956,656603,681568,695454,733103,744520,722609,731412,735230,750865,2369229,55.01
2,UNKNOWN,628735,500941,548595,641114,613507,670570,697188,720080,728064,719239,711457,737983,740887,8658360,22.05,225392,189693,204628,224798,221517,234517,240740,252523,257134,251356,250825,254738,260343,734463,17.05


### Zwigato_check

In [44]:
cx_data7 = f'''
with base as (
    select 
        customer_id,
        taxi_regularity_segment
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        count(distinct order_id) orders
    from 
        orders.order_logs_fact
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        and order_status = 'dropped'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    ),
    
    appo as (
    
    select
        userid,
            /* 
            case 
                when regexp_like(apps_installed, 'jira|miro|zoho|teams||intune|figma|slack|confluence|webex|github') = true 
                then 'office-apps' 
            else 'no-office-apps' end office_tag,  
        
        CASE 
            WHEN apps_installed LIKE '%uber%' OR apps_installed LIKE '%ola%' OR apps_installed LIKE '%blusmart%' OR apps_installed LIKE '%nammayatri%' OR apps_installed LIKE '%in drive%' 
            OR apps_installed LIKE '%quick ride%' OR apps_installed LIKE '%jugnoo%' 
        THEN 'RHA' 
            else 'NRHA' end as RHA_Check,
        */

        CASE 
            when apps_installed LIKE '%zomato%' OR apps_installed LIKE '%swiggy%' 
        THEN 'Zwigato' 
            else 'Non-Zwigato' end as Zwigato_check

            
    from 
        datasets.customer_apps_installed_snapshot
    )
    
    
    select
        coalesce(Zwigato_check, 'UNKNOWN') Zwigato_check,
        sum(case when week_start_date = '20241216' then orders end ) w01,
        sum(case when week_start_date = '20241223' then orders end ) w02,
        sum(case when week_start_date = '20241230' then orders end ) w03,
        sum(case when week_start_date = '20250106' then orders end ) w04,
        sum(case when week_start_date = '20250113' then orders end ) w05,
        sum(case when week_start_date = '20250120' then orders end ) w06,
        sum(case when week_start_date = '20250127' then orders end ) w07,
        sum(case when week_start_date = '20250203' then orders end ) w08,
        sum(case when week_start_date = '20250210' then orders end ) w09,
        sum(case when week_start_date = '20250217' then orders end ) w10,
        sum(case when week_start_date = '20250224' then orders end ) w11,
        sum(case when week_start_date = '20250303' then orders end ) w12,
        sum(case when week_start_date = '20250310' then orders end ) w13,

        sum(orders.orders) as overall,
        100.00*sum(orders)/sum(sum(orders))over() as pct,
        
        count(distinct case when week_start_date = '20241216' then orders.customer_id end ) C_w01,
        count(distinct case when week_start_date = '20241223' then orders.customer_id end ) C_w02,
        count(distinct case when week_start_date = '20241230' then orders.customer_id end ) C_w03,
        count(distinct case when week_start_date = '20250106' then orders.customer_id end ) C_w04,
        count(distinct case when week_start_date = '20250113' then orders.customer_id end ) C_w05,
        count(distinct case when week_start_date = '20250120' then orders.customer_id end ) C_w06,
        count(distinct case when week_start_date = '20250127' then orders.customer_id end ) C_w07,
        count(distinct case when week_start_date = '20250203' then orders.customer_id end ) C_w08,
        count(distinct case when week_start_date = '20250210' then orders.customer_id end ) C_w09,
        count(distinct case when week_start_date = '20250217' then orders.customer_id end ) C_w10,
        count(distinct case when week_start_date = '20250224' then orders.customer_id end ) C_w11,
        count(distinct case when week_start_date = '20250303' then orders.customer_id end ) C_w12,
        count(distinct case when week_start_date = '20250310' then orders.customer_id end ) C_w13,
        
        count(distinct orders.customer_id) as C_overall,
        100.00*count(distinct orders.customer_id)/sum(count(distinct orders.customer_id))over() as c_pct

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    left join
        appo
        on base.customer_id = appo.userid
    group by 1
'''
cx_data7 = pd.read_sql(cx_data7, conn1)
cx_data7.head()

,Zwigato_check,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,Zwigato,1552516,1329657,1442052,1566445,1514891,1626588,1672537,1752345,1769714,1733222,1738978,1789485,1813418,21301848,54.26,608349,543764,581229,607402,603987,626487,639773,675489,686151,666253,677017,682021,696431,2119252,49.20
1,Non-Zwigato,661155,626085,654864,679390,674628,693167,710457,755756,770350,751345,749247,768816,806347,9301607,23.69,302282,291986,304189,309905,316281,315955,321884,343793,352524,341107,344802,346383,363551,1453373,33.74
2,UNKNOWN,628735,500941,548595,641114,613507,670570,697188,720080,728064,719239,711457,737983,740887,8658360,22.05,225392,189693,204628,224798,221517,234517,240740,252523,257134,251356,250825,254738,260343,734463,17.05


### Office Usecases

In [45]:
cx_data8 = f'''
with base as (
    select 
        customer_id,
        taxi_regularity_segment
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        count(distinct order_id) orders
    from 
        orders.order_logs_fact
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        and order_status = 'dropped'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    ),
    
    appo as (
    
    select
        userid,
            case 
                when regexp_like(apps_installed, 'jira|miro|zoho|teams|intune|figma|slack|confluence|webex|github') = true 
                then 'office-apps' 
            else 'no-office-apps' end office_tag    
        from 
        datasets.customer_apps_installed_snapshot
    )
    
    
    select
        coalesce(office_tag, 'UNKNOWN') office_tag,
        sum(case when week_start_date = '20241216' then orders end ) w01,
        sum(case when week_start_date = '20241223' then orders end ) w02,
        sum(case when week_start_date = '20241230' then orders end ) w03,
        sum(case when week_start_date = '20250106' then orders end ) w04,
        sum(case when week_start_date = '20250113' then orders end ) w05,
        sum(case when week_start_date = '20250120' then orders end ) w06,
        sum(case when week_start_date = '20250127' then orders end ) w07,
        sum(case when week_start_date = '20250203' then orders end ) w08,
        sum(case when week_start_date = '20250210' then orders end ) w09,
        sum(case when week_start_date = '20250217' then orders end ) w10,
        sum(case when week_start_date = '20250224' then orders end ) w11,
        sum(case when week_start_date = '20250303' then orders end ) w12,
        sum(case when week_start_date = '20250310' then orders end ) w13,

        sum(orders.orders) as overall,
        100.00*sum(orders)/sum(sum(orders))over() as pct,
        
        count(distinct case when week_start_date = '20241216' then orders.customer_id end ) C_w01,
        count(distinct case when week_start_date = '20241223' then orders.customer_id end ) C_w02,
        count(distinct case when week_start_date = '20241230' then orders.customer_id end ) C_w03,
        count(distinct case when week_start_date = '20250106' then orders.customer_id end ) C_w04,
        count(distinct case when week_start_date = '20250113' then orders.customer_id end ) C_w05,
        count(distinct case when week_start_date = '20250120' then orders.customer_id end ) C_w06,
        count(distinct case when week_start_date = '20250127' then orders.customer_id end ) C_w07,
        count(distinct case when week_start_date = '20250203' then orders.customer_id end ) C_w08,
        count(distinct case when week_start_date = '20250210' then orders.customer_id end ) C_w09,
        count(distinct case when week_start_date = '20250217' then orders.customer_id end ) C_w10,
        count(distinct case when week_start_date = '20250224' then orders.customer_id end ) C_w11,
        count(distinct case when week_start_date = '20250303' then orders.customer_id end ) C_w12,
        count(distinct case when week_start_date = '20250310' then orders.customer_id end ) C_w13,
        
        count(distinct orders.customer_id) as C_overall,
        100.00*count(distinct orders.customer_id)/sum(count(distinct orders.customer_id))over() as c_pct
      
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    left join
        appo
        on base.customer_id = appo.userid
    group by 1
    order by 1 
'''
cx_data8 = pd.read_sql(cx_data8, conn1)
cx_data8.head()

,office_tag,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,UNKNOWN,628735,500941,548595,641114,613507,670570,697188,720080,728064,719239,711457,737983,740887,8658360,22.05,225392,189693,204628,224798,221517,234517,240740,252523,257134,251356,250825,254738,260343,734463,17.05
1,no-office-apps,1469808,1355749,1432430,1492242,1479678,1528872,1558964,1649768,1679274,1638774,1634308,1671615,1735623,20327105,51.77,616557,585519,613803,623132,636123,637323,647204,687838,704220,680637,688335,689067,717155,2556038,59.34
2,office-apps,743863,599993,664486,753593,709841,790883,824030,858333,860790,845793,853917,886686,884142,10276350,26.17,294074,250231,271615,294175,284145,305119,314453,331444,334455,326723,333484,339337,342827,1016587,23.60


In [63]:
cx_data8

,office_tag,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,UNKNOWN,628735,500941,548595,641114,613507,670570,697188,720080,728064,719239,711457,737983,740887,8658360,22.05,225392,189693,204628,224798,221517,234517,240740,252523,257134,251356,250825,254738,260343,734463,17.05
1,no-office-apps,1469808,1355749,1432430,1492242,1479678,1528872,1558964,1649768,1679274,1638774,1634308,1671615,1735623,20327105,51.77,616557,585519,613803,623132,636123,637323,647204,687838,704220,680637,688335,689067,717155,2556038,59.34
2,office-apps,743863,599993,664486,753593,709841,790883,824030,858333,860790,845793,853917,886686,884142,10276350,26.17,294074,250231,271615,294175,284145,305119,314453,331444,334455,326723,333484,339337,342827,1016587,23.60


### Age Group

In [46]:
cx_data9 = f'''
with base as (
    select 
        customer_id,
        taxi_regularity_segment
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        count(distinct order_id) orders
    from 
        orders.order_logs_fact
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        and order_status = 'dropped'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    ),
    
    age as (
    
    select
        customer_id,
        age_group
    from 
        hive.experiments_internal.customer_predicated_age_immutable
    )
    
    
    select
        coalesce(age_group, 'UNKNOWN') age_group,
        sum(case when week_start_date = '20241216' then orders end ) w01,
        sum(case when week_start_date = '20241223' then orders end ) w02,
        sum(case when week_start_date = '20241230' then orders end ) w03,
        sum(case when week_start_date = '20250106' then orders end ) w04,
        sum(case when week_start_date = '20250113' then orders end ) w05,
        sum(case when week_start_date = '20250120' then orders end ) w06,
        sum(case when week_start_date = '20250127' then orders end ) w07,
        sum(case when week_start_date = '20250203' then orders end ) w08,
        sum(case when week_start_date = '20250210' then orders end ) w09,
        sum(case when week_start_date = '20250217' then orders end ) w10,
        sum(case when week_start_date = '20250224' then orders end ) w11,
        sum(case when week_start_date = '20250303' then orders end ) w12,
        sum(case when week_start_date = '20250310' then orders end ) w13,
        
        sum(orders.orders) as overall,
        100.00*sum(orders)/sum(sum(orders))over() as pct,
        
        count(distinct case when week_start_date = '20241216' then orders.customer_id end ) C_w01,
        count(distinct case when week_start_date = '20241223' then orders.customer_id end ) C_w02,
        count(distinct case when week_start_date = '20241230' then orders.customer_id end ) C_w03,
        count(distinct case when week_start_date = '20250106' then orders.customer_id end ) C_w04,
        count(distinct case when week_start_date = '20250113' then orders.customer_id end ) C_w05,
        count(distinct case when week_start_date = '20250120' then orders.customer_id end ) C_w06,
        count(distinct case when week_start_date = '20250127' then orders.customer_id end ) C_w07,
        count(distinct case when week_start_date = '20250203' then orders.customer_id end ) C_w08,
        count(distinct case when week_start_date = '20250210' then orders.customer_id end ) C_w09,
        count(distinct case when week_start_date = '20250217' then orders.customer_id end ) C_w10,
        count(distinct case when week_start_date = '20250224' then orders.customer_id end ) C_w11,
        count(distinct case when week_start_date = '20250303' then orders.customer_id end ) C_w12,
        count(distinct case when week_start_date = '20250310' then orders.customer_id end ) C_w13,
        
        count(distinct orders.customer_id) as C_overall,
        100.00*count(distinct orders.customer_id)/sum(count(distinct orders.customer_id))over() as c_pct

      
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    left join
        age
        on base.customer_id = age.customer_id
    group by 1
    order by 1
'''
cx_data9 = pd.read_sql(cx_data9, conn1)
cx_data9.head(10)

,age_group,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,21-25,509873,456853,499531,522643,521816,534225,548593,577548,568701,543323,547984,552993,573680,6957763,17.72,221700,204949,222803,225712,231473,230347,234649,246403,240743,226812,231866,228207,236646,856025,19.87
1,26-30,478597,428342,465846,491231,472351,499278,517382,543320,535370,513466,520225,529711,535016,6530135,16.63,197954,181369,195244,201546,197729,203538,208682,218557,214853,205502,208597,209633,213100,695672,16.15
2,31-35,406287,359524,379016,411077,397284,425582,436771,452920,444428,428970,424217,434698,437711,5438485,13.85,164710,150446,156961,165656,163672,170124,173073,179911,175374,168453,167928,169014,171452,565804,13.14
3,36-45,601464,522688,550682,601850,583558,630934,645042,666384,649062,631614,615268,628920,629256,7956722,20.27,239093,219579,227105,237909,239250,247494,252018,261069,251325,243161,240417,239086,241598,830405,19.28
4,Above-45,129640,111252,118444,129918,127440,137886,141381,147803,142852,139833,135707,138500,140540,1741196,4.43,51701,46953,48516,50929,51920,53971,55519,57923,55508,53495,53118,52805,53749,189354,4.40
5,Below-21,39670,35839,38924,40314,39501,40691,41541,43975,41438,39661,39714,40278,41576,523122,1.33,16738,15815,16991,16901,17531,17571,17763,18581,17480,16483,16819,16504,17096,63587,1.48
6,UNKNOWN,676875,542185,593068,689916,661076,721729,749472,796231,886277,906939,916567,971184,1002873,10114392,25.76,244127,206332,222426,243452,240210,253914,260693,289361,340526,344810,353899,367893,386684,1106241,25.68


In [65]:
cx_data9

,age_group,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,21-25,509873,456853,499531,522643,521816,534225,548593,577548,568701,543323,547984,552993,573680,6957763,17.72,221700,204949,222803,225712,231473,230347,234649,246403,240743,226812,231866,228207,236646,856025,19.87
1,26-30,478597,428342,465846,491231,472351,499278,517382,543320,535370,513466,520225,529711,535016,6530135,16.63,197954,181369,195244,201546,197729,203538,208682,218557,214853,205502,208597,209633,213100,695672,16.15
2,31-35,406287,359524,379016,411077,397284,425582,436771,452920,444428,428970,424217,434698,437711,5438485,13.85,164710,150446,156961,165656,163672,170124,173073,179911,175374,168453,167928,169014,171452,565804,13.14
3,36-45,601464,522688,550682,601850,583558,630934,645042,666384,649062,631614,615268,628920,629256,7956722,20.27,239093,219579,227105,237909,239250,247494,252018,261069,251325,243161,240417,239086,241598,830405,19.28
4,Above-45,129640,111252,118444,129918,127440,137886,141381,147803,142852,139833,135707,138500,140540,1741196,4.43,51701,46953,48516,50929,51920,53971,55519,57923,55508,53495,53118,52805,53749,189354,4.40
5,Below-21,39670,35839,38924,40314,39501,40691,41541,43975,41438,39661,39714,40278,41576,523122,1.33,16738,15815,16991,16901,17531,17571,17763,18581,17480,16483,16819,16504,17096,63587,1.48
6,UNKNOWN,676875,542185,593068,689916,661076,721729,749472,796231,886277,906939,916567,971184,1002873,10114392,25.76,244127,206332,222426,243452,240210,253914,260693,289361,340526,344810,353899,367893,386684,1106241,25.68


### Office Use-Cases

In [47]:
cx_data11 = f'''
with base as (
    select 
        customer_id,
        taxi_regularity_segment
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    use_case as (
    
    select
        hex_12,
        primary_tag
    from 
        experiments_internal.combined_geo_usecase_hex_12_level
    where 
        current_city = 'Bangalore'
        and primary_tag = 'office'
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        case 
        when pickup_location_hex_12 in (select distinct hex_12 from use_case) then 'office'
        when drop_location_hex_12 in (select distinct hex_12 from use_case) then 'office'
        else 'non-office' end office_use_case,
        count(distinct order_id) orders
    from 
        orders.order_logs_snapshot
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        and order_status = 'dropped'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2,3
    )
    
    select
        coalesce(office_use_case, 'non-office') office_use_case,
        sum(case when week_start_date = '20241216' then orders end ) w01,
        sum(case when week_start_date = '20241223' then orders end ) w02,
        sum(case when week_start_date = '20241230' then orders end ) w03,
        sum(case when week_start_date = '20250106' then orders end ) w04,
        sum(case when week_start_date = '20250113' then orders end ) w05,
        sum(case when week_start_date = '20250120' then orders end ) w06,
        sum(case when week_start_date = '20250127' then orders end ) w07,
        sum(case when week_start_date = '20250203' then orders end ) w08,
        sum(case when week_start_date = '20250210' then orders end ) w09,
        sum(case when week_start_date = '20250217' then orders end ) w10,
        sum(case when week_start_date = '20250224' then orders end ) w11,
        sum(case when week_start_date = '20250303' then orders end ) w12,
        sum(case when week_start_date = '20250310' then orders end ) w13,
        
        sum(orders.orders) as overall,
        100.00*sum(orders)/sum(sum(orders))over() as pct,

        count(distinct case when week_start_date = '20241216' then orders.customer_id end ) C_w01,
        count(distinct case when week_start_date = '20241223' then orders.customer_id end ) C_w02,
        count(distinct case when week_start_date = '20241230' then orders.customer_id end ) C_w03,
        count(distinct case when week_start_date = '20250106' then orders.customer_id end ) C_w04,
        count(distinct case when week_start_date = '20250113' then orders.customer_id end ) C_w05,
        count(distinct case when week_start_date = '20250120' then orders.customer_id end ) C_w06,
        count(distinct case when week_start_date = '20250127' then orders.customer_id end ) C_w07,
        count(distinct case when week_start_date = '20250203' then orders.customer_id end ) C_w08,
        count(distinct case when week_start_date = '20250210' then orders.customer_id end ) C_w09,
        count(distinct case when week_start_date = '20250217' then orders.customer_id end ) C_w10,
        count(distinct case when week_start_date = '20250224' then orders.customer_id end ) C_w11,
        count(distinct case when week_start_date = '20250303' then orders.customer_id end ) C_w12,
        count(distinct case when week_start_date = '20250310' then orders.customer_id end ) C_w13,

        count(distinct orders.customer_id) as C_overall,
        100.00*count(distinct orders.customer_id)/sum(count(distinct orders.customer_id))over() as c_pct
        
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1
    order by 1
'''
cx_data11 = pd.read_sql(cx_data11, conn1)
cx_data11.head(10)

,office_use_case,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,non-office,2447354,2178490,2351068,2478663,2434847,2563861,2644707,2774820,2806257,2741052,2749270,2814135,2886677,33871201,86.27,1041304,953795,1016143,1045690,1051460,1077286,1102591,1166184,1186804,1149655,1165312,1170078,1206913,4126851,73.44
1,office,395052,278193,294443,408286,368179,426464,435475,453361,461871,462754,450412,482149,473975,5390614,13.73,242563,185005,196080,244612,229942,253719,257489,271786,278891,274949,272522,285146,287877,1492522,26.56


In [69]:
cx_data11

,office_use_case,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,non-office,2447354,2178490,2351068,2478663,2434847,2563861,2644707,2774820,2806257,2741052,2749270,2814135,2886677,33871201,86.27,1041304,953795,1016143,1045690,1051460,1077286,1102591,1166184,1186804,1149655,1165312,1170078,1206913,4126851,73.44
1,office,395052,278193,294443,408286,368179,426464,435475,453361,461871,462754,450412,482149,473975,5390614,13.73,242563,185005,196080,244612,229942,253719,257489,271786,278891,274949,272522,285146,287877,1492522,26.56
